# SHAP Explanations and Trading Decision Quality — Statistical Analysis

**Design summary (as I understand it — fix in Cell 1 if wrong):**
- Within-subjects: each participant does 20 trials, 10 `explained` (SHAP shown) and 10 `unexplained`.
- `group` = counterbalance order (which condition came first).
- Outcome measures per trial: `decision` (buy/sell/hold), `confidence_rating` (1–4), `response_time_seconds`.
- No ground-truth market outcome column was in the sample provided, so **decision quality is proxied by alignment between `decision` and `signal_shown`** (up→buy, down→sell, flat→hold = "aligned"). Replace `score_decision()` in Cell 4 if you have real outcome data — that's the stronger DV.

**Pipeline:** load → clean/dedupe → determine completion → build analysis dataset → descriptives → within-subject inferential tests (accuracy, confidence, RT) → mixed-effects models → calibration analysis → export tables for the paper.


In [ ]:
# Cell 1 — Imports & config
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', context='paper')

# ---- CONFIG: fill in your Google Sheet details ----
# Get these from the sheet's URL: .../d/<SHEET_ID>/edit#gid=<GID>
SHEET_ID = "PASTE_YOUR_SHEET_ID_HERE"
GID_TRIALS = "PASTE_TRIALS_TAB_GID"
GID_SESSIONS = "PASTE_SESSIONS_TAB_GID"

def gsheet_csv_url(sheet_id, gid):
    return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

EXPECTED_TRIALS_PER_SESSION = 20
CONDITIONS = ["explained", "unexplained"]


In [ ]:
# Cell 2 — Load data
trials_raw = pd.read_csv(gsheet_csv_url(SHEET_ID, GID_TRIALS))
sessions_raw = pd.read_csv(gsheet_csv_url(SHEET_ID, GID_SESSIONS))

# If the public-CSV export doesn't work (private sheet), instead export both tabs
# to CSV manually and load with:
# trials_raw = pd.read_csv('/mnt/user-data/uploads/trials.csv')
# sessions_raw = pd.read_csv('/mnt/user-data/uploads/sessions.csv')

for df in (trials_raw, sessions_raw):
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
    df['participant_id'] = df['participant_id'].astype(str).str.strip()

print("trials_raw:", trials_raw.shape)
print("sessions_raw:", sessions_raw.shape)
trials_raw.head()


In [ ]:
# Cell 3 — Clean trials: dedupe exact re-submissions
# Some rows are duplicate submissions of the same trial (same participant/trial_index/
# decision/etc, fired moments apart -- see participant 02, trial_index 13 in your sample).
# Keep the FIRST submission for each (participant_id, trial_index).

dupe_key = ['participant_id', 'trial_index']
n_before = len(trials_raw)

trials = (trials_raw
          .sort_values('timestamp_utc')
          .drop_duplicates(subset=dupe_key, keep='first')
          .reset_index(drop=True))

print(f"Dropped {n_before - len(trials)} duplicate trial submissions "
      f"({n_before} -> {len(trials)} rows)")

# Sanity check: trial_index should be 0-19, condition should split 10/10
print(trials.groupby('condition')['trial_index'].agg(['min', 'max', 'count']))


In [ ]:
# Cell 4 — Resolve session status & completion per participant
# sessions has one row per session EVENT ('started', possibly later 'completed').
# group is sometimes logged as 'unknown' on the completed row (data bug) --
# use the group value from the 'started' row as authoritative.

sess = sessions_raw.copy()

group_lookup = (sess[sess['group'] != 'unknown']
                 .sort_values('timestamp_utc')
                 .groupby('participant_id')['group']
                 .first())

completion = (sess.groupby('participant_id')['status']
                  .apply(lambda s: 'completed' if 'completed' in set(s) else 'started')
                  .rename('session_status'))

n_trials_done = trials.groupby('participant_id').size().rename('n_trials')

participants = (pd.DataFrame(group_lookup).join(completion).join(n_trials_done)
                   .rename(columns={'group': 'group'})
                   .reset_index())
participants['n_trials'] = participants['n_trials'].fillna(0).astype(int)

participants['is_complete'] = (
    (participants['session_status'] == 'completed') &
    (participants['n_trials'] == EXPECTED_TRIALS_PER_SESSION)
)

print(participants['is_complete'].value_counts())
participants.head(10)


In [ ]:
# Cell 5 — Build the analysis dataset (completers only)
# NOTE: decide here whether to run the primary analysis on completers only
# (cleanest) and a secondary/sensitivity analysis on all available trials
# (larger n, uses partial data). Both are set up below.

completers = participants.loc[participants['is_complete'], 'participant_id']

trials_full = trials.merge(participants[['participant_id', 'group', 'session_status', 'is_complete']],
                            on='participant_id', how='left', suffixes=('', '_sess'))

analysis_df = trials_full[trials_full['participant_id'].isin(completers)].copy()
partial_df = trials_full[~trials_full['participant_id'].isin(completers)].copy()

print(f"Completers: {participants['is_complete'].sum()} participants, "
      f"{len(analysis_df)} trials")
print(f"Non-completers: {(~participants['is_complete']).sum()} participants, "
      f"{len(partial_df)} partial trials (excluded from primary analysis)")

analysis_df['condition'] = pd.Categorical(analysis_df['condition'], categories=CONDITIONS)
analysis_df.head()


In [ ]:
# Cell 6 — Derived variables: decision alignment (accuracy proxy) & calibration
# ADAPT score_decision() if you have real ground-truth market outcomes instead.

SIGNAL_TO_EXPECTED_DECISION = {'up': 'buy', 'down': 'sell', 'flat': 'hold'}

def score_decision(row):
    return int(row['decision'] == SIGNAL_TO_EXPECTED_DECISION.get(row['signal_shown']))

analysis_df['aligned'] = analysis_df.apply(score_decision, axis=1)

# Confidence calibration: confidence_shown is 0-1 (model's stated confidence),
# confidence_rating is a 1-4 Likert (participant's stated confidence).
# Rescale participant rating to 0-1 for a comparable calibration-error metric.
analysis_df['confidence_rating_norm'] = (analysis_df['confidence_rating'] - 1) / 3
analysis_df['calibration_error'] = (analysis_df['confidence_rating_norm'] -
                                     analysis_df['confidence_shown']).abs()

analysis_df[['participant_id', 'condition', 'signal_shown', 'decision', 'aligned',
             'confidence_shown', 'confidence_rating_norm', 'calibration_error',
             'response_time_seconds']].head()


In [ ]:
# Cell 7 — Descriptive statistics
desc = (analysis_df.groupby('condition')
        .agg(n_trials=('aligned', 'size'),
             accuracy=('aligned', 'mean'),
             mean_confidence_rating=('confidence_rating', 'mean'),
             mean_calibration_error=('calibration_error', 'mean'),
             median_rt=('response_time_seconds', 'median'),
             mean_rt=('response_time_seconds', 'mean')))
print(desc)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.barplot(data=analysis_df, x='condition', y='aligned', errorbar='ci', ax=axes[0])
axes[0].set_title('Decision-Signal Alignment (accuracy proxy)')
sns.barplot(data=analysis_df, x='condition', y='confidence_rating', errorbar='ci', ax=axes[1])
axes[1].set_title('Confidence Rating')
sns.boxplot(data=analysis_df, x='condition', y='response_time_seconds', ax=axes[2])
axes[2].set_title('Response Time (s)')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 8 — Participant-level aggregation for paired tests
# Within-subject design -> aggregate each participant x condition, then compare paired.

participant_cond = (analysis_df.groupby(['participant_id', 'group', 'condition'], observed=True)
                     .agg(accuracy=('aligned', 'mean'),
                          mean_confidence=('confidence_rating', 'mean'),
                          calibration_error=('calibration_error', 'mean'),
                          median_rt=('response_time_seconds', 'median'))
                     .reset_index())

wide_acc = participant_cond.pivot(index='participant_id', columns='condition', values='accuracy')
wide_conf = participant_cond.pivot(index='participant_id', columns='condition', values='mean_confidence')
wide_cal = participant_cond.pivot(index='participant_id', columns='condition', values='calibration_error')
wide_rt = participant_cond.pivot(index='participant_id', columns='condition', values='median_rt')

wide_acc.head()


In [ ]:
# Cell 9 — Normality checks (decide parametric vs non-parametric)
for name, wide in [('accuracy', wide_acc), ('confidence', wide_conf),
                    ('calibration_error', wide_cal), ('RT', wide_rt)]:
    diff = (wide['explained'] - wide['unexplained']).dropna()
    stat, p = stats.shapiro(diff)
    print(f"{name:20s} Shapiro-Wilk on paired differences: W={stat:.3f}, p={p:.4f}"
          f"  -> {'looks normal' if p > .05 else 'NOT normal, prefer Wilcoxon'}")


In [ ]:
# Cell 10 — Primary within-subject tests: explained vs unexplained
def paired_report(wide, label):
    d = wide.dropna()
    diff = d['explained'] - d['unexplained']
    t, p_t = stats.ttest_rel(d['explained'], d['unexplained'])
    w, p_w = stats.wilcoxon(d['explained'], d['unexplained'])
    dz = diff.mean() / diff.std(ddof=1)  # Cohen's dz for paired designs
    print(f"--- {label} (n={len(d)}) ---")
    print(f"  Mean explained={d['explained'].mean():.3f}, unexplained={d['unexplained'].mean():.3f}, "
          f"diff={diff.mean():.3f}")
    print(f"  Paired t-test:  t={t:.3f}, p={p_t:.4f}")
    print(f"  Wilcoxon:       W={w:.3f}, p={p_w:.4f}")
    print(f"  Cohen's dz:     {dz:.3f}\n")
    return {'label': label, 'n': len(d), 'mean_explained': d['explained'].mean(),
            'mean_unexplained': d['unexplained'].mean(), 't': t, 'p_ttest': p_t,
            'W': w, 'p_wilcoxon': p_w, 'dz': dz}

results = []
results.append(paired_report(wide_acc, 'Decision accuracy (signal alignment)'))
results.append(paired_report(wide_conf, 'Confidence rating'))
results.append(paired_report(wide_cal, 'Calibration error'))
results.append(paired_report(wide_rt, 'Response time (median, s)'))

results_table = pd.DataFrame(results)
results_table


In [ ]:
# Cell 11 — Mixed-effects models (trial-level, controls for order/group)
# Logistic GLMM for accuracy; LMM for confidence and RT. Random intercept per participant.

# Accuracy (binary) -- mixed logit via GEE-style random-effects approx (statsmodels MixedLM
# doesn't do logit GLMM directly; use BinomialBayesMixedGLM for a proper mixed logistic model)
import statsmodels.api as sm

glmm_acc = sm.BinomialBayesMixedGLM.from_formula(
    "aligned ~ condition + group",
    vc_formulas={"participant": "0 + C(participant_id)"},
    data=analysis_df
).fit_vb()
print(glmm_acc.summary())


In [ ]:
# Cell 12 — LMM for confidence and response time
lmm_conf = smf.mixedlm("confidence_rating ~ condition + group", analysis_df,
                        groups=analysis_df["participant_id"]).fit()
print(lmm_conf.summary())


In [ ]:
# Cell 13 — Response time (log-transform: RTs are typically right-skewed)
analysis_df['log_rt'] = np.log(analysis_df['response_time_seconds'])
lmm_rt = smf.mixedlm("log_rt ~ condition + group", analysis_df,
                      groups=analysis_df["participant_id"]).fit()
print(lmm_rt.summary())


In [ ]:
# Cell 14 — Order/carryover check (does condition order interact with the effect?)
# Since condition is nested within a fixed trial order (0-9 explained, 10-19 unexplained
# per your sample) rather than counterbalanced per trial, check whether 'group' captures
# presentation order and test the condition x group interaction as an order-effect check.
lmm_acc_order = smf.mixedlm("aligned ~ condition * group", analysis_df,
                             groups=analysis_df["participant_id"]).fit()
print(lmm_acc_order.summary())


In [ ]:
# Cell 15 — Sensitivity analysis on non-completers (optional, robustness check)
if len(partial_df) > 0:
    partial_df['condition'] = pd.Categorical(partial_df['condition'], categories=CONDITIONS)
    partial_df['aligned'] = partial_df.apply(score_decision, axis=1)
    sens_desc = partial_df.groupby('condition')['aligned'].agg(['size', 'mean'])
    print("Non-completer trials, descriptive only (underpowered for inference):")
    print(sens_desc)
else:
    print("No partial/non-completer data to report.")


In [ ]:
# Cell 16 — Export summary tables for the paper
results_table.to_csv('/mnt/user-data/outputs/paired_test_results.csv', index=False)
desc.to_csv('/mnt/user-data/outputs/descriptive_stats_by_condition.csv')
participant_cond.to_csv('/mnt/user-data/outputs/participant_level_data.csv', index=False)
analysis_df.to_csv('/mnt/user-data/outputs/trial_level_clean_data.csv', index=False)

print("Exported: paired_test_results.csv, descriptive_stats_by_condition.csv, "
      "participant_level_data.csv, trial_level_clean_data.csv")


## Next step

Once you run this and share the outputs (console output, tables, or the exported CSVs), send them back
and I'll help you:
- interpret effect sizes and significance in the context of your hypothesis,
- write this up in APA-style results-section prose,
- flag anything that looks off (e.g., low completer n, non-normal distributions, order effects),
- decide whether the accuracy proxy needs replacing with real outcome data.
